# Real-Time Video Background Blur via Parallelized Gaussian Mixture Model

**CSC14116 — Applied Parallel Programming** | HCMUS

**Pipeline:** Sequential Python -> Numba CPU Parallel -> CUDA GPU

The per-pixel model (`GMM_*_MOG2`) is a from-scratch port of OpenCV's
`BackgroundSubtractorMOG2` (Zivkovic 2004). Section 2 measures it against
OpenCV — bit-exact on synthetic input and, on x86-64, on real 8-bit video too —
which is what lets us compare the three implementations against a single ground
truth.

## §1 — Environment & Setup

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from settings import MOG2_N_COMPONENTS
from gmm.mog2_common import opencv_reference, to_planar
from utils import benchmark as bm
from pipeline import detect_platform, available_models, best_model, make_pipeline, gpu_name

pinfo = detect_platform()
models = available_models()
print(pinfo)
print('models:', list(models))

In [ ]:
VIDEO_PATH = '../input.mp4'   # any clip; falls back to synthetic frames
N_FRAMES = 60

if os.path.exists(VIDEO_PATH):
    frames_full = bm.load_video(VIDEO_PATH, max_frames=N_FRAMES)
else:
    print('no video found — using synthetic frames')
    frames_full = bm.synthetic_frames(N_FRAMES, 480, 640)

RES = {'480p': (640, 480), '720p': (1280, 720), '1080p': (1920, 1080)}
frames = {name: bm.resize_frames(frames_full, w, h) for name, (w, h) in RES.items()}
print({k: (len(v), v[0].shape) for k, v in frames.items()})

## §2 — Correctness against OpenCV MOG2

`BackgroundSubtractorMOG2` is the reference. Our kernel reproduces it: the
adaptive update (`k = alpha / w_new`), the per-pixel active-mode count, the
insertion sort, the complexity-reduction pruning, the separate `Tb` / `Tg` / `TB`
thresholds, and the shadow test. The learning rate follows OpenCV's warm-up ramp
`1/min(2*nframes, history)`.

On synthetic sequences the masks match bit for bit. On real 8-bit video the
result is platform-dependent: exact on x86-64 Linux (0 of 9,216,000 pixels
differ on a Colab T4 runtime), ~0.002% differing on macOS arm64, where that
OpenCV build contracts `acc += d*d` into an FMA and rounds once where we round
twice. The three MOG2 models agree with each other on masks exactly.

In [ ]:
test_frames = frames['480p'][:30]
mog2 = opencv_reference()
cv_masks = [mog2.apply(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)) for f in test_frames]

Model = models['numba_cpu']
model = Model(test_frames[0], n_components=MOG2_N_COMPONENTS, color=False)
our_masks = [model.step(to_planar(f, color=False))[0].copy() for f in test_frames]

m = [bm.mask_metrics(o, c) for o, c in zip(our_masks, cv_masks)]
nd = sum(int((a != b).sum()) for a, b in zip(cv_masks, our_masks))
tot = sum(a.size for a in cv_masks)
inter = sum(int(((a == 255) & (b == 255)).sum()) for a, b in zip(cv_masks, our_masks))
union = sum(int(((a == 255) | (b == 255)).sum()) for a, b in zip(cv_masks, our_masks))
print(f'differing px   : {nd}/{tot}  ({nd/tot*100:.5f}%)')
print('pixel accuracy :', np.mean([x['accuracy'] for x in m]))
print('IoU (aggregate):', inter / max(union, 1))
print('IoU (per-frame):', np.mean([x['iou'] for x in m]), '  <- small early masks dominate')
print('F1  (last frame):', m[-1]['f1'])
print('background img max abs error:',
      np.abs(model.background_image().astype(int)
             - mog2.getBackgroundImage().astype(int).reshape(model.background_image().shape)).max())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(cv2.cvtColor(test_frames[-1], cv2.COLOR_BGR2RGB)); ax[0].set_title('input')
ax[1].imshow(cv_masks[-1], cmap='gray'); ax[1].set_title('OpenCV MOG2')
ax[2].imshow(our_masks[-1], cmap='gray'); ax[2].set_title('ours (GMM_CPU_NUMBA_MOG2)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## §3 — Version 1: Sequential Python (`GMM_CPU_MOG2`)

Pure Python, one pixel at a time. `gmm/cpu/GMM_cpu_mog2.py` is the specification
the other two models are transliterated from.

```
for each pixel:
    for mode in active modes (descending weight):
        w = (1-a)*w + prune                     # complexity reduction
        if not matched yet:
            dist2 = |mean - pixel|^2
            if cumulative_w < TB and dist2 < Tb*var:  background = True
            if dist2 < Tg*var:                        # this mode fits
                w += a;  k = a / w                    # k from the NEW weight
                mean += -k * (mean - pixel)
                var   = clamp(var + k*(dist2 - var), varMin, varMax)
                bubble this mode up while it outweighs its neighbour
        if w < -prune: drop the mode
    renormalise the active weights
    if nothing matched: append (or replace weakest) with (a, pixel, varInit)
```

In [ ]:
small = bm.resize_frames(test_frames[:5], 160, 120)
row_seq = bm.benchmark(models['sequential'], small, n_warmup=0, n_measure=1)
print(row_seq)

## §4 — Version 2: Numba CPU Parallel (`GMM_CPU_NUMBA_MOG2`)

`@njit(parallel=True)` with `prange` over rows — each row owns its pixels' state,
so there is no synchronisation at all. Output is bit-identical to §3.

In [ ]:
rows_cpu = {name: bm.benchmark(models['numba_cpu'], fr[:20], n_measure=2)
            for name, fr in frames.items()}
for k, v in rows_cpu.items():
    print(f"{k:6s} {v['total']*1e3:8.2f} ms  {v['fps']:7.1f} FPS   "
          f"(gmm {v['gmm']*1e3:.2f} | morph {v['morph']*1e3:.2f} | blur {v['blur_composite']*1e3:.2f})")

In [ ]:
# separable (2 x 1D) vs naive 2D convolution
print(bm.benchmark_blur_variants(frames['480p']))

## §5 — Version 3: CUDA GPU (`GMM_CUDA_MOG2`)

One thread per pixel. What the GPU version does beyond a naive port:

| Optimisation | Where |
|---|---|
| Planar state `(K,H,W)` / `(K,C,H,W)`, so neighbouring threads hit neighbouring addresses | `mog2_common.MOG2Base` |
| Persistent device state — the model never leaves the GPU between frames | `GMM_CUDA_MOG2` |
| Separable blur: 2×15 taps instead of 15×15 | `utils/blur_cuda.py` |
| Shared-memory tiling with halo on **both** blur passes | same |
| Kernel fusion: the vertical pass writes the composite directly, so the blurred frame is never stored | `blur_v_composite_kernel` |
| Pinned host buffers + 2 CUDA streams — the upload of frame *i* overlaps the compute of frame *i-1* | `CUDAPipeline.process_stream` |
| Morphological opening (erode + dilate) to clean the mask | `erode_kernel`, `dilate_kernel` |

Measured on a Colab T4: **40.2 FPS at 1080p** (877x the sequential reference),
114 FPS at 720p, 371 FPS at 480p. The separable + tiled blur is 22.6x faster
than the naive 2D convolution at 1080p. Streaming adds ~10% at 480p/720p but
costs ~6% at 1080p, where compute already dominates the upload it hides.

*Runs where an NVIDIA GPU is present (Colab T4). On a CPU-only machine set
`NUMBA_ENABLE_CUDASIM=1` and run `tests/test_correctness.py` to validate the
kernels — correctness only, the timings below need a real GPU.*

In [ ]:
rows_gpu, rows_gpu_stream = {}, {}
if 'cuda' in models and pinfo['has_cuda']:
    from pipeline import gpu_name
    print(gpu_name())
    for name, fr in frames.items():
        rows_gpu[name] = bm.benchmark(models['cuda'], fr, n_measure=3)
        rows_gpu_stream[name] = bm.benchmark_streamed(models['cuda'], fr)
        print(name, f"sync {rows_gpu[name]['fps']:.1f} FPS | "
                    f"streamed {rows_gpu_stream[name]['fps']:.1f} FPS")
    print('blur variants:', bm.benchmark_blur_variants(frames['1080p'], use_cuda=True))
else:
    print('CUDA not available on this machine — run this notebook on Colab (T4).')

## §6 — Benchmark summary

In [ ]:
import pandas as pd

rows = [{'Model': 'GMM_CPU_MOG2', 'Resolution': row_seq['resolution'],
         'ms/frame': row_seq['total']*1e3, 'FPS': row_seq['fps']}]
for name, v in rows_cpu.items():
    rows.append({'Model': 'GMM_CPU_NUMBA_MOG2', 'Resolution': name,
                 'ms/frame': v['total']*1e3, 'FPS': v['fps']})
for name, v in rows_gpu.items():
    rows.append({'Model': 'GMM_CUDA_MOG2', 'Resolution': name,
                 'ms/frame': v['total']*1e3, 'FPS': v['fps']})
for name, v in rows_gpu_stream.items():
    rows.append({'Model': 'GMM_CUDA_MOG2 (streamed)', 'Resolution': name,
                 'ms/frame': v['total']*1e3, 'FPS': v['fps']})
df = pd.DataFrame(rows)
display(df)

In [ ]:
piv = df[df.Model != 'GMM_CPU_MOG2'].pivot_table(index='Resolution', columns='Model', values='FPS')
piv = piv.reindex([r for r in ['480p', '720p', '1080p'] if r in piv.index])
ax = piv.plot.bar(figsize=(9, 5), rot=0)
ax.axhline(30, ls='--', c='k', lw=1); ax.text(0.02, 31, '30 FPS target', fontsize=9)
ax.set_ylabel('FPS'); ax.set_title('Throughput by model and resolution')
plt.tight_layout(); plt.show()

## §7 — Demo output

In [ ]:
name = best_model(models)
p = make_pipeline(models[name], frames['480p'][0], n_components=MOG2_N_COMPONENTS)
for f in frames['480p']:
    out, mask, t = p.process(f)
plt.figure(figsize=(15, 4))
plt.imshow(cv2.cvtColor(bm.display_side_by_side(frames['480p'][-1], mask, np.asarray(out)),
                        cv2.COLOR_BGR2RGB))
plt.axis('off'); plt.title(f'input | mask | blurred output  ({p.NAME})')
plt.show()

## §8 — Discussion

### Why the GMM kernel is memory-bound
Per pixel we touch `K=5 × (weight, variance) + K × C means` — 15 float32 for
grayscale (60 B) — and do ~30 FLOPs. Arithmetic intensity is ~0.5 FLOP/byte, far
below the T4's ridge point, so the kernel is bandwidth-bound. The planar layout
is what makes those loads coalesce: thread `x` and thread `x+1` read
`weights[k, y, x]` and `weights[k, y, x+1]`, which are adjacent.

### Why the blur benefits more from the GPU
A 15×15 window is 225 taps per output pixel, and adjacent outputs share 14/15 of
their neighbourhood. Two things exploit that: **separability** cuts 225 taps to
30, and **shared-memory tiling** removes the redundant global reads that remain.
Fusing the vertical pass with the composite removes one full-frame write and one
full-frame read on top.

### Warp divergence
The mode loop is data-dependent — pixels with 1 active mode and pixels with 5
diverge, and the insertion sort runs a different number of swaps per thread.
This is inherent to MOG2. In practice most pixels settle at 1–2 modes, so the
divergence is bounded and shrinks as the model converges.

### Foreground persistence
A stopped object stays foreground for `ln(TB) / ln(1-alpha)` frames — the *old*
background mode has to decay below the background ratio `TB`. At the default
`alpha = 1/500` that is ~53 frames (~1.8 s at 30 FPS). It is **not**
`TB × history`. Lower the learning rate if you want people who sit still to stay
sharp for longer; `tests/test_correctness.py` checks this against OpenCV at
three learning rates.

### Relation to the earlier models
`GMM_CPU` / `GMM_CPU_NUMBA` / `GMM_CUPY_V0` / `GMM_CUPY_V1` implement
Stauffer-Grimson with a fixed K and a constant-alpha EMA. The `*_MOG2` models
keep the same planar state layout and the same class contract, but replace the
update rule with Zivkovic's, which is what buys the adaptive mode count and the
match with OpenCV.

### Platform portability
- **macOS Apple Silicon:** no CUDA and no CuPy. `NUMBA_ENABLE_CUDASIM=1` runs the
  CUDA kernels on the CPU, so correctness can still be validated locally.
- **Google Colab T4:** full CUDA path.